In [4]:
# IMPORTS
import torch
import torch.nn as nn
from Binn import BINN
import data_handling as dh
import binn_training as bt
import pandas as pd
import os 

In [ ]:
# GLOBALS
ALL_CELLTYPES = [0,1,2,3,4,5,6,7,8]
EPOCHS = 20

In [ ]:
print("Reading adata...")
train_adata, test_adata, acollection = dh.read_adata(ALL_CELLTYPES, filepath=, train_size=0.8)

In [ ]:
print("Getting dataloader...")
train_loader, test_loader = dh.create_dataloaders(train_adata, test_adata)

In [ ]:
# Behövs??
#print("Concatenating data...")
#adata_conc = dh.data_concatenate(acollection)

In [ ]:
base_path = os.getcwd()
mask_paths = [f"/PathwayData/MaskMatrixLayers/mg_200_mc_200_mhvg1000/oligo_exc3_exc2_vasc_immune_astro_inhi_opcs_exc1_layer_{i}_mask.csv" 
              for i in range(5)]

# Mask matrix files
masks = {}
for i, mask_path in enumerate(mask_paths):
    df = pd.read_csv(base_path + mask_path, index_col=0)
    masks.update({f"df{i}": df})
    print(f"Matrix {i} shape: {df.shape}")

print(masks.keys())

In [ ]:
# Extract pure number representation matrices from masks
mask_list = [masks[mask].to_numpy() for mask in masks]

# Starting amount of features
in_features = masks["df0"].shape[0]

# Extract layer dimensions
layers_list = [masks[mask].shape[1] for mask in masks]

print(f"input features: {in_features}")
print(f"layer list: {layers_list}")

In [ ]:
# Init BINN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BINN(in_features=in_features,
                  layers_list=layers_list,
                  mask_list=mask_list).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Step through the training
for epoch in range(EPOCHS):
    train_loss, train_acc = bt.train_binn(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = bt.test_binn(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1} / {EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}")
    print("-" * 30)
